# Training Analysis & Rollout Visualization

This notebook pulls **all training data** from W&B, displays:
1. **Summary table** — runtime (minutes), total steps, final reward
2. **Reward curves** with ± std shading
3. **Cartesian coordinates** from a replayed rollout of a selected run

In [7]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

## 1 — Configuration

Set your W&B entity, project, and the **run IDs** you want to analyse.  
Set `ROLLOUT_RUN_ID` to the single run from which you want Cartesian coordinates.

In [10]:
import wandb
from itertools import islice

api = wandb.Api()
runs = api.runs("weissma6-zhaw-school-of-engineering/UR10_pick_ppo", order="-created_at")

for run in islice(runs, 10):
    print(run.id, run.name, run.state)

SilGrip_random_qpos_lr0.0011_512_20260204_205936_2660 SilGrip_random_qpos_lr0.0011_512_20260204_205936_2660 finished
SilGrip_random_qpos_lr0.0010_512_20260204_205833_190 SilGrip_random_qpos_lr0.0010_512_20260204_205833_190 finished
SilGrip_random_qpos_lr0.0009_512_20260204_205820_8953 SilGrip_random_qpos_lr0.0009_512_20260204_205820_8953 finished
SilGrip_random_qpos_lr0.0007_512_20260204_203338_2100 SilGrip_random_qpos_lr0.0007_512_20260204_203338_2100 finished
SilGrip_random_qpos_lr0.0006_512_20260204_203339_7445 SilGrip_random_qpos_lr0.0006_512_20260204_203339_7445 finished
SilGrip_random_qpos_lr0.0008_512_20260204_203337_3901 SilGrip_random_qpos_lr0.0008_512_20260204_203337_3901 finished
SilGrip_random_qpos_lr0.0005_512_20260204_203255_503 SilGrip_random_qpos_lr0.0005_512_20260204_203255_503 finished
SilGrip_random_qpos_lr0.0004_512_20260204_203249_2381 SilGrip_random_qpos_lr0.0004_512_20260204_203249_2381 finished
SilGrip_random_qpos_lr0.0011_512_20260202_225440_3902 SilGrip_random

In [15]:
# ──────────────── EDIT THESE ────────────────
ENTITY  = "weissma6-zhaw-school-of-engineering"
PROJECT_PANDA = "UR10_pick_ppo"  # or "UR10_pick_ppo"
PROJECT_PANDA = "panda_pick_ppo"  # or "UR10_pick_ppo"

# Dict of display-label → W&B run ID for reward curves
RUNS_UR10 = {
    "SilGrip_random_qpos_lr0.0008_512_20260204_203337_3901": "SilGrip_random_qpos_lr0.0008_512_20260204_203337_3901",
}
RUNS_PANDA = {
    "Panda_default_seed6": "2hcb0tlw",
    "default_seed6": "umm9hky6",
}


# Environment name used during training
ENV_NAME = "UR10PickCube"  # or "UR10PickCube"

# Rollout settings
ROLLOUT_SEED = 0
RENDER_EVERY = 1
# ────────────────────────────────────────────

## 2 — Fetch training histories from W&B

In [14]:
api = wandb.Api()

histories = {}   # label → DataFrame
summaries = []   # list of dicts for the summary table

for label, run_id in RUNS.items():
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")

    # ── History (reward curves) ──
    hist = run.history(
        keys=["eval/episode_reward", "eval/episode_reward_std", "training/num_steps"],
        pandas=True,
    ).dropna(subset=["eval/episode_reward"])
    histories[label] = hist

    # ── Run metadata ──
    created  = pd.Timestamp(run.created_at)
    # heartbeat_at is the last time the run reported; approximates end time
    finished = pd.Timestamp(run.summary.get("_timestamp", run.created_at), unit="s")
    runtime_min = run.summary.get("_runtime", 0) / 60.0  # W&B logs _runtime in seconds

    total_steps = (
        int(hist["training/num_steps"].max())
        if "training/num_steps" in hist.columns and len(hist) > 0
        else int(run.summary.get("training/num_steps", 0))
    )
    final_reward = (
        float(hist["eval/episode_reward"].iloc[-1])
        if len(hist) > 0
        else float(run.summary.get("eval/episode_reward", float("nan")))
    )

    summaries.append({
        "Run":            label,
        "ID":             run_id,
        "State":          run.state,
        "Runtime (min)":  round(runtime_min, 2),
        "Total Steps":    total_steps,
        "Final Reward":   round(final_reward, 2),
    })

    print(f"✓ {label:30s}  steps={total_steps:>12,}  runtime={runtime_min:6.1f} min  reward={final_reward:.2f}")

summary_df = pd.DataFrame(summaries)
display(summary_df)

CommError: Could not find run <Run weissma6-zhaw-school-of-engineering/panda_pick_ppo/SilGrip_random_qpos_lr0.0008_512_20260204_203337_3901 (not found)>

## 3 — Reward curves (mean ± std)

In [ ]:
n = len(histories)
colors = plt.cm.tab10.colors[:n]

fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=True, squeeze=False)
axes = axes.flatten()

for ax, (label, df), color in zip(axes, histories.items(), colors):
    steps = df["_step"] if "_step" in df.columns else df.index
    mean  = df["eval/episode_reward"]
    std   = df.get("eval/episode_reward_std", pd.Series(0, index=df.index))

    ax.plot(steps, mean, color=color, linewidth=2, label="Mean Reward")
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.2, label="± Std Dev")

    ax.set_xlabel("Environment Steps")
    ax.set_title(label, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
    )

axes[0].set_ylabel("Episode Reward")
plt.suptitle("Training Reward Curves", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("reward_comparison.png", bbox_inches="tight", dpi=300)
plt.savefig("reward_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()

: 

## 4 — All-runs overlay (single plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for (label, df), color in zip(histories.items(), colors):
    steps = df["_step"] if "_step" in df.columns else df.index
    mean  = df["eval/episode_reward"]
    std   = df.get("eval/episode_reward_std", pd.Series(0, index=df.index))

    ax.plot(steps, mean, color=color, linewidth=2, label=label)
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.15)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Reward")
ax.set_title("All Runs — Reward Overlay", fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)
plt.tight_layout()
plt.savefig("reward_overlay.png", bbox_inches="tight", dpi=300)
plt.show()

: 

## 5 — Runtime & Steps bar charts

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

labels = summary_df["Run"]
x = np.arange(len(labels))

# Runtime
ax1.barh(x, summary_df["Runtime (min)"], color="steelblue")
ax1.set_yticks(x)
ax1.set_yticklabels(labels)
ax1.set_xlabel("Runtime (minutes)")
ax1.set_title("Training Runtime", fontweight="bold")
for i, v in enumerate(summary_df["Runtime (min)"]):
    ax1.text(v + 0.3, i, f"{v:.1f}", va="center", fontsize=9)

# Steps
ax2.barh(x, summary_df["Total Steps"], color="coral")
ax2.set_yticks(x)
ax2.set_yticklabels(labels)
ax2.set_xlabel("Total Training Steps")
ax2.set_title("Training Steps", fontweight="bold")
ax2.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)

plt.tight_layout()
plt.savefig("runtime_steps.png", bbox_inches="tight", dpi=300)
plt.show()

: 

---
## 6 — Cartesian Coordinates from Rollout Replay

Downloads the saved model artifact from `ROLLOUT_RUN_ID`, replays the policy in the environment,
and extracts the **end-effector** and **cube** Cartesian positions at every timestep.

In [ ]:
import os, glob
from flax import serialization

# ── Download model artifact ──
rollout_run = api.run(f"{ENTITY}/{PROJECT}/{ROLLOUT_RUN_ID}")

# Find the policy artifact logged by this run
artifacts = list(rollout_run.logged_artifacts())
model_arts = [a for a in artifacts if a.type == "model"]
assert len(model_arts) > 0, f"No model artifacts found for run {ROLLOUT_RUN_ID}"

art = model_arts[-1]  # latest
art_dir = art.download()  # downloads to ./artifacts/...
print(f"Downloaded artifact '{art.name}' → {art_dir}")
print("Contents:", os.listdir(art_dir))

params_file = glob.glob(os.path.join(art_dir, "*.msgpack"))
assert params_file, "No .msgpack params file found in artifact!"
params_path = params_file[0]
print(f"Params file: {params_path}")

: 

In [ ]:
import jax
import jax.numpy as jnp
import functools
from mujoco_playground import registry, wrapper
from mujoco_playground.config import manipulation_params
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo

# ── Recreate env + network to restore params ──
env = registry.load(ENV_NAME)
env_cfg = registry.get_default_config(ENV_NAME)
episode_length = int(getattr(env_cfg, "episode_length", 1000))

# Retrieve network_factory config from the run's W&B config
run_config = dict(rollout_run.config)
nf_params = run_config.get("network_factory", {})
if isinstance(nf_params, dict) and nf_params:
    if isinstance(nf_params.get("policy_hidden_layer_sizes"), list):
        nf_params["policy_hidden_layer_sizes"] = tuple(nf_params["policy_hidden_layer_sizes"])
    if isinstance(nf_params.get("value_hidden_layer_sizes"), list):
        nf_params["value_hidden_layer_sizes"] = tuple(nf_params["value_hidden_layer_sizes"])
    network_factory = functools.partial(ppo_networks.make_ppo_networks, **nf_params)
else:
    network_factory = ppo_networks.make_ppo_networks

print(f"Env: {ENV_NAME}  |  episode_length: {episode_length}")
print(f"Network factory params: {nf_params}")

: 

In [ ]:
# ── Build the network and deserialise params ──
# We need the environment wrapped the same way training wraps it
wrapped_env = wrapper.wrap_for_brax_training(env)

# Build network structure (obs_size, action_size come from the wrapped env)
obs_size = wrapped_env.observation_size
action_size = wrapped_env.action_size

ppo_net = network_factory(
    observation_size=obs_size,
    action_size=action_size,
    preprocess_observations_fn=None,
)

# Deserialise the saved params
with open(params_path, "rb") as f:
    raw_bytes = f.read()

# Create a dummy params tree to use as the target structure
rng = jax.random.PRNGKey(0)
dummy_obs = jnp.zeros((1, obs_size))
dummy_key = jax.random.PRNGKey(42)

init_params = ppo_net.policy_network.init(dummy_key)
restored_params = serialization.from_bytes(init_params, raw_bytes)

print(f"✓ Params restored  (obs_size={obs_size}, action_size={action_size})")

: 

In [ ]:
# ── Run the rollout and collect Cartesian positions ──
make_policy = ppo_net.make_policy
policy = jax.jit(make_policy(restored_params, deterministic=True))

jit_reset = jax.jit(env.reset)
jit_step  = jax.jit(env.step)

rng = jax.random.PRNGKey(ROLLOUT_SEED)
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)

# Storage
rollout_states = [state]
ee_positions = []     # end-effector XYZ
cube_positions = []   # cube XYZ  (if present)
rewards = []

for t in range(episode_length):
    rng, act_rng = jax.random.split(rng)
    out = policy(state.obs, act_rng)
    ctrl = out[0] if isinstance(out, tuple) else out
    ctrl = jnp.asarray(ctrl)

    state = jit_step(state, ctrl)
    rollout_states.append(state)
    rewards.append(float(jnp.asarray(state.reward)))

    # ── Extract Cartesian coords from pipeline_state ──
    # MuJoCo Playground exposes body positions via pipeline_state
    ps = state.pipeline_state

    # xpos shape: (n_bodies, 3)  — index depends on your model
    # We grab all body positions and later pick the relevant ones
    if hasattr(ps, "x"):  # Brax MJX style
        all_pos = np.asarray(ps.x.pos)
    elif hasattr(ps, "xpos"):  # Classic MuJoCo bindings
        all_pos = np.asarray(ps.xpos)
    else:
        all_pos = None

    if all_pos is not None:
        ee_positions.append(all_pos.copy())

ee_positions = np.array(ee_positions)  # (T, n_bodies, 3)
rewards = np.array(rewards)

print(f"Rollout done: {len(rewards)} steps")
print(f"Body positions shape: {ee_positions.shape}  (timesteps, n_bodies, 3)")
print(f"Cumulative reward: {rewards.sum():.2f}")

: 

In [ ]:
# ── Identify which body indices to plot ──
# Print body names so you can pick the right ones
# (MuJoCo model body names are stored in the MjModel)
try:
    mj_model = env.mj_model if hasattr(env, "mj_model") else env.sys.mj_model
    body_names = [mj_model.body(i).name for i in range(mj_model.nbody)]
    print("Body indices and names:")
    for i, name in enumerate(body_names):
        print(f"  [{i:2d}] {name}")
except Exception as e:
    print(f"Could not read body names: {e}")
    body_names = [f"body_{i}" for i in range(ee_positions.shape[1])]
    print("Using generic body names. Check your model to pick the right indices.")

: 

In [ ]:
# ──────────── EDIT THESE to match your model ────────────
# After seeing the body name list above, set the correct indices
EE_BODY_IDX   = -2   # end-effector / gripper tip (often second-to-last body)
CUBE_BODY_IDX = -1   # manipulated object (often last body)
# ────────────────────────────────────────────────────────

ee_xyz   = ee_positions[:, EE_BODY_IDX, :]   # (T, 3)
cube_xyz = ee_positions[:, CUBE_BODY_IDX, :]  # (T, 3)

ee_name   = body_names[EE_BODY_IDX]   if EE_BODY_IDX   < len(body_names) else f"body {EE_BODY_IDX}"
cube_name = body_names[CUBE_BODY_IDX] if CUBE_BODY_IDX < len(body_names) else f"body {CUBE_BODY_IDX}"

print(f"End-effector body: [{EE_BODY_IDX}] '{ee_name}'")
print(f"Cube body:         [{CUBE_BODY_IDX}] '{cube_name}'")

: 

## 7 — Plot Cartesian Trajectories

In [ ]:
timesteps = np.arange(len(ee_xyz))

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
coord_labels = ["X", "Y", "Z"]

for i, (ax, cl) in enumerate(zip(axes, coord_labels)):
    ax.plot(timesteps, ee_xyz[:, i],   linewidth=1.5, label=f"{ee_name} {cl}")
    ax.plot(timesteps, cube_xyz[:, i], linewidth=1.5, label=f"{cube_name} {cl}", linestyle="--")
    ax.set_ylabel(f"{cl} position (m)")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestep")
axes[0].set_title(f"Cartesian Coordinates — run {ROLLOUT_RUN_ID}", fontweight="bold")
plt.tight_layout()
plt.savefig("cartesian_coords_xyz.png", bbox_inches="tight", dpi=300)
plt.show()

: 

In [ ]:
# ── 3D trajectory plot ──
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(ee_xyz[:, 0], ee_xyz[:, 1], ee_xyz[:, 2],
        linewidth=1.5, label=ee_name, color="tab:blue")
ax.plot(cube_xyz[:, 0], cube_xyz[:, 1], cube_xyz[:, 2],
        linewidth=1.5, label=cube_name, color="tab:orange", linestyle="--")

# Mark start and end
ax.scatter(*ee_xyz[0],  s=80, c="green",  marker="o", label="EE start", zorder=5)
ax.scatter(*ee_xyz[-1], s=80, c="red",    marker="X", label="EE end",   zorder=5)
ax.scatter(*cube_xyz[0],  s=80, c="green",  marker="s", zorder=5)
ax.scatter(*cube_xyz[-1], s=80, c="red",    marker="D", zorder=5)

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title(f"3D Trajectory — run {ROLLOUT_RUN_ID}", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("cartesian_3d_trajectory.png", bbox_inches="tight", dpi=300)
plt.show()

: 

In [ ]:
# ── Save coordinates to CSV for further use ──
coords_df = pd.DataFrame({
    "timestep":     timesteps,
    "ee_x":         ee_xyz[:, 0],
    "ee_y":         ee_xyz[:, 1],
    "ee_z":         ee_xyz[:, 2],
    "cube_x":       cube_xyz[:, 0],
    "cube_y":       cube_xyz[:, 1],
    "cube_z":       cube_xyz[:, 2],
    "step_reward":  rewards,
})
coords_df.to_csv("rollout_cartesian_coords.csv", index=False)
print("Saved rollout_cartesian_coords.csv")
display(coords_df.head(10))

: 